# Stablecoin StressBench — Evaluation & Leaderboard

This notebook evaluates all trained models and produces the benchmark leaderboard:
- ML metrics: AUROC, AUPRC, F1, Brier score
- Economic metrics: net bps captured, hit rate, Sharpe ratio, max drawdown
- Leaderboard ranking

In [ ]:
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.metrics import roc_auc_score

from stressbench.experiments.feature_sets import FEATURE_SETS
from stressbench.evaluation.backtest import run_backtest
from stressbench.evaluation.leaderboard import build_leaderboard, print_leaderboard
from stressbench.models.baselines import LogisticBaseline, LastValueBaseline
from stressbench.models.tree_models import RandomForestWrapper, LGBMWrapper

DATASET_PATH = Path("../data/gold/dataset.parquet")
NET_COL    = "net_profit_bps_q10000"
NET_THRESH = 10.0                      # c = 10 bps (Eq. 5)
FEATURE_SET = "price_plus_book"

df = pl.read_parquet(DATASET_PATH)
feature_cols = FEATURE_SETS[FEATURE_SET]

def load_split(df, split, net_col, net_thresh, feature_cols):
    sdf = df.filter(pl.col("split") == split)
    X_raw = sdf.select(feature_cols).to_numpy().astype(np.float32)
    nan_mask = np.isnan(X_raw)
    if nan_mask.any():
        col_med = np.where(np.isnan(np.nanmedian(X_raw, axis=0)), 0.0,
                           np.nanmedian(X_raw, axis=0))
        X = np.where(nan_mask, col_med[None, :], X_raw)
    else:
        X = X_raw
    y_net = sdf[net_col].to_numpy().astype(np.float64)
    y_exec = (y_net > net_thresh).astype(np.int8)
    return X, y_exec, y_net

X_train, y_train, y_net_tr = load_split(df, "train",      NET_COL, NET_THRESH, feature_cols)
X_val,   y_val,   y_net_v  = load_split(df, "validation", NET_COL, NET_THRESH, feature_cols)
X_test,  y_test,  y_net    = load_split(df, "test",       NET_COL, NET_THRESH, feature_cols)

print(f"Test split: {X_test.shape}, pos rate: {y_test.mean():.4f}")
oracle_mask = y_net > NET_THRESH
print(f"Oracle windows: {oracle_mask.sum()} / {len(y_net)}  "
      f"mean net_bps: {y_net[oracle_mask].mean():.1f} bps")

def calibrate_threshold(y_proba, y_net_val, n_cand=17, min_trades=25):
    best_t, best_total = 0.5, -np.inf
    for t in np.linspace(0.05, 0.95, n_cand):
        sig = y_proba > t
        if sig.sum() < min_trades:
            continue
        total = float(y_net_val[sig].sum())
        if total > best_total:
            best_total, best_t = total, float(t)
    return best_t

model_configs = [
    ("NoTrade",    None),
    ("LogisticReg", LogisticBaseline()),
    ("RF",          RandomForestWrapper(task="classification", n_estimators=100)),
    ("LightGBM",   LGBMWrapper(task="classification")),
]

results = []
for name, model in model_configs:
    if model is None:
        results.append({"model": name, "n_trades": 0, "net_bps": 0.0,
                         "signal": np.zeros(len(y_test), dtype=bool)})
        print(f"{name:15s}  trades=   0  net_bps=  0.0")
        continue
    model.fit(X_train, y_train)
    val_proba  = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    theta = calibrate_threshold(val_proba, y_net_v)
    signal = test_proba > theta
    n_trades = int(signal.sum())
    net_bps  = float(y_net[signal].mean()) if n_trades > 0 else 0.0
    try:
        auroc = roc_auc_score(y_test, test_proba)
    except Exception:
        auroc = float("nan")
    results.append({"model": name, "n_trades": n_trades,
                     "net_bps": round(net_bps, 1), "signal": signal})
    print(f"{name:15s}  trades={n_trades:5d}  net_bps={net_bps:+.1f}  auroc={auroc:.3f}")

oracle_bps = float(y_net[oracle_mask].mean())
print(f"Oracle          trades={oracle_mask.sum():5d}  net_bps={oracle_bps:+.1f}  (ceiling)")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLORS = {
    "NoTrade":    "#999999",
    "LogisticReg": "#d62728",
    "RF":          "#1f77b4",
    "LightGBM":   "#ff7f0e",
}

fig, ax = plt.subplots(figsize=(12, 5))

for r in results:
    signal = r["signal"]
    if signal.sum() == 0:
        ax.axhline(0, color=COLORS.get(r["model"], "black"),
                   ls="--", lw=1.2, label=r["model"])
        continue
    # Cumulative net bps for triggered trades in time order
    trade_returns = y_net[signal]
    cum = np.cumsum(trade_returns)
    ax.plot(range(len(cum)), cum, lw=1.5,
            color=COLORS.get(r["model"], None), label=r["model"])

# Oracle ceiling
oracle_idx = np.where(y_net > 10)[0]
oracle_ret = y_net[oracle_idx]
ax.plot(range(len(oracle_ret)), np.cumsum(oracle_ret),
        color="gold", lw=2, ls="-", label="Oracle (ceiling)")

ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_title("Cumulative Net bps per Triggered Trade (SVB test split)")
ax.set_xlabel("Trade index (time order)")
ax.set_ylabel("Cumulative net bps")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig("../results/paper/figures/figure_3_cumulative_pnl.png", dpi=150)
plt.show()
print("Saved figure_3_cumulative_pnl.png")
